In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile

zip_path = '/content/drive/MyDrive/archive (1).zip'
extract_path = '/content/customer_support'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted successfully")

import os
for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(os.path.join(root, file))

Extracted successfully
/content/customer_support/sample.csv
/content/customer_support/twcs/twcs.csv


In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/customer_support/twcs/twcs.csv')

# Filter to Amazon Help
df_amazon = df[df['author_id'] == 'AmazonHelp']

print("Amazon Help shape:", df_amazon.shape)

df_amazon.head()

Amazon Help shape: (169840, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
181,269,AmazonHelp,False,Wed Nov 22 09:23:01 +0000 2017,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272.0
184,273,AmazonHelp,False,Wed Nov 22 09:40:27 +0000 2017,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271.0
186,275,AmazonHelp,False,Wed Nov 22 10:06:26 +0000 2017,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274.0
234,324,AmazonHelp,False,Wed Nov 22 09:06:00 +0000 2017,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325.0
321,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0


In [4]:
# Step 1: Get all inbound customer tweets (the original complaints)
df_inbound = df[df['inbound'] == True]

# Step 2: Join inbound tweets with Amazon's replies
# We match: Amazon's reply.in_response_to_tweet_id == customer's tweet_id
df_pairs = pd.merge(
    df_inbound,
    df_amazon,
    left_on='tweet_id',
    right_on='in_response_to_tweet_id',
    suffixes=('_customer', '_amazon')
)

# Step 3: Keep only the useful columns
df_pairs = df_pairs[[
    'tweet_id_customer', 'text_customer',
    'tweet_id_amazon', 'text_amazon',
    'created_at_customer', 'created_at_amazon'
]]

print("Total customer-Amazon pairs:", df_pairs.shape)
df_pairs.head()

Total customer-Amazon pairs: (168814, 6)


,tweet_id_customer,text_customer,tweet_id_amazon,text_amazon,created_at_customer,created_at_amazon
0,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,Wed Nov 22 09:30:36 +0000 2017,Wed Nov 22 09:40:27 +0000 2017
1,274,@AmazonHelp こちらこそありがとうございました。,275,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,Wed Nov 22 09:44:04 +0000 2017,Wed Nov 22 10:06:26 +0000 2017
2,272,amazonのfireTVstickが見れない😢,269,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,Wed Nov 22 09:14:39 +0000 2017,Wed Nov 22 09:23:01 +0000 2017
3,325,amazonプライムビデオ、再生エラーが多いです,324,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,Wed Nov 22 08:55:35 +0000 2017,Wed Nov 22 09:06:00 +0000 2017
4,616,@AmazonHelp 3 different people have given 3 di...,618,@115820 We'd like to take a further look into ...,Tue Oct 31 23:22:08 +0000 2017,Tue Oct 31 23:28:00 +0000 2017


In [5]:
def is_mostly_english(text):
    if not isinstance(text, str):
        return False
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return ascii_chars / len(text) > 0.9

df_english = df_pairs[df_pairs['text_customer'].apply(is_mostly_english)].reset_index(drop=True)

print("English-only pairs:", df_english.shape)
df_english.head()

English-only pairs: (158368, 6)


,tweet_id_customer,text_customer,tweet_id_amazon,text_amazon,created_at_customer,created_at_amazon
0,616,@AmazonHelp 3 different people have given 3 di...,618,@115820 We'd like to take a further look into ...,Tue Oct 31 23:22:08 +0000 2017,Tue Oct 31 23:28:00 +0000 2017
1,617,Way to drop the ball on customer service @1158...,615,@115820 I'm sorry we've let you down! Without ...,Tue Oct 31 22:16:32 +0000 2017,Tue Oct 31 22:29:00 +0000 2017
2,621,@115823 I want my amazon payments account CLOS...,620,@115822 I am unable to affect your account via...,Tue Oct 31 22:19:34 +0000 2017,Tue Oct 31 22:28:34 +0000 2017
3,623,"@AmazonHelp Okay, danke für die Info",625,@115824 Wir haben zu danken. Schönen Abend noc...,Tue Oct 31 22:32:07 +0000 2017,Tue Oct 31 22:34:32 +0000 2017
4,624,"@115825 also, beim Addams Family-Film in Prime...",622,"@115824 Hi, wir erhalten die Filme/Serien so v...",Tue Oct 31 22:12:37 +0000 2017,Tue Oct 31 22:28:00 +0000 2017


In [6]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'@\w+', '', text)          # remove @mentions
    text = re.sub(r'http\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'\s+', ' ', text).strip()   # collapse extra whitespace
    return text

df_english['clean_customer'] = df_english['text_customer'].apply(clean_text)
df_english['clean_amazon'] = df_english['text_amazon'].apply(clean_text)

df_english[['clean_customer', 'clean_amazon']].head(10)

,clean_customer,clean_amazon
0,3 different people have given 3 different answ...,We'd like to take a further look into this wit...
1,Way to drop the ball on customer service so pi...,I'm sorry we've let you down! Without providin...
2,I want my amazon payments account CLOSED. dm m...,I am unable to affect your account via Twitter...
3,"Okay, danke für die Info",Wir haben zu danken. Schönen Abend noch. ^JS
4,"also, beim Addams Family-Film in Prime sind Bi...","Hi, wir erhalten die Filme/Serien so vom jewei..."
5,Yeah this is crazy we’re less than a week away...,Thanks for your patience. ^KM
6,How about you guys figure out my Xbox One X pr...,I'm sorry for the wait. You'll receive an emai...
7,my package was ‘accidentally’ opened.. 4 items...,I'm sorry your order arrived in this condition...
8,why is my order at my local courier for the la...,I'm sorry for the wait. Please reach out to us...
9,"Thanks for the style advice, look ...I think? ...",Alexa says both styles are working for you! My...


In [7]:
import random

sample = df_english['clean_customer'].sample(20, random_state=42).tolist()
for i, msg in enumerate(sample):
    print(f"{i+1}. {msg}\n")

1. my order was supposed to arrive tmrw but Purolator has informed me amazon didn't deliver it yet and still has to fly from BC-ON

2. Xbox

3. I am not signing Into your website. This is nothing to do with me or my account.

4. These return instructions could be written more clearly

5. Ja, dachte es würden Staub in den Lautsprecher gefallen sein. Ich sag mal so ab halber Lautstärke nimmt das knacken zu :/

6. So wheres my "guaranteed" 2 day delivery? When can I expect delivery or will it even be delivered? Always late..

7. I already contacted via chat and was told to wait 24 hours. That was about 18 hours ago.

8. It's a work based enquiry and can't find paperwork/unable to contact manager.

9. Material sending back that too Prepaid, which kind of service is this.

10. please initiate my refund against my order detail Order date 05-Oct-2017 Order # 407-7959994-9600365 Order total 414.77

11. :Order1__credit_card__ shows delivered but the order is not received yet.Customer service as

In [ ]:
sample2 = df_english['clean_customer'].sample(20, random_state=99).tolist()
for i, msg in enumerate(sample2):
    print(f"{i+1}. {msg}\n")

1. Is this your fax number? 1-206-922-5821 they are requesting my e-mail address by e-mailing to my e-mail address😬

2. Terrible is the situation! You all biggest frauds and criminals of the world!

3. Can you follow me to DM you.

4. Order No 407-4058875-4231560. Flase report #EcomExpress, delivery not attempted and got the message. Where is my package?

5. I can send a dm with the order number

6. Some of ur sellers are goons and u looted money delivered duplicate product and no customer service facility

7. sehe ich das richtig? Das #Mate10Lite verkauft ihr nicht selber? Nur über Drittanbieter?

8. Thanks to inaptitude, I could be the owner of world’s most expensive Polaroid film-sent to me in lieu of Panasonic lens #amazon

9. tomorrow I will file a case in Delhi court against Order # 405-7739204-775556. Be ready

10. Someone opened my amazon package, and retaped it. I don't know if it was the delivery driver or someone in the neighborhood

11. still waiting on an order from Novemb

In [ ]:
import pandas as pd

# Sample 280 messages for hand-labeling (buffer above the 250 target)
label_sample = df_english[['tweet_id_customer', 'clean_customer', 'clean_amazon']].sample(
    n=280, random_state=7
).reset_index(drop=True)

# Add empty columns for manual labeling
label_sample['intent'] = ""
label_sample['escalate'] = ""   # will fill with TRUE or FALSE
label_sample['notes'] = ""      # optional, for anything ambiguous

# Save to CSV
label_sample.to_csv('/content/drive/MyDrive/golden_set_unlabeled.csv', index=False)

print("Saved:", label_sample.shape)

Saved: (280, 6)


In [8]:
!pip install openai --quiet

from openai import OpenAI
from google.colab import userdata

# Pull your secret key safely (never hardcode it)
NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

In [ ]:
INTENT_LABELS = [
    "delivery_delay",
    "order_not_received",
    "refund_return",
    "damaged_or_wrong_item",
    "account_access",
    "payment_issue",
    "prime_subscription",
    "product_content_request",
    "customer_service_escalation",
    "technical_issue",
    "follow_up_acknowledgment",
    "non_english",
    "general_other"
]

def classify_intent(message):
    prompt = f"""You are classifying customer support messages sent to Amazon on Twitter.

Choose exactly ONE label from this list that best matches the message:
{', '.join(INTENT_LABELS)}

Message: "{message}"

Reply with ONLY the label, nothing else. No explanation, no punctuation."""

    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=20,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )

    label = response.choices[0].message.content.strip()
    return label

In [ ]:
test_message = df_english['clean_customer'].iloc[0]
print("Message:", test_message)
print("Predicted intent:", classify_intent(test_message))

Message: 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day
Predicted intent: order_not_received


In [ ]:

import pandas as pd

df_golden = pd.read_excel('/content/drive/MyDrive/golden_set_labeled.xlsx')

print(df_golden.shape)
df_golden.head()

(280, 6)


,tweet_id_customer,clean_customer,clean_amazon,intent,escalate,notes
0,1366973,Two of the items don't have a delivery date ev...,I'm sorry for the wait! You'll receive a confi...,delivery_delay,True,NaN
1,1254604,I haven’t been charged yet either,What delivery date were you given in your conf...,payment_issue,True,NaN
2,2697196,Merci de votre réponse aussi rapide. Je vais d...,"Je vous en prie, bonne soirée à vous aussi. ^BR",non_english,False,NaN
3,1456410,I have provided my details last night via chat...,We wouldn't be able to access your Amazon acco...,customer_service_escalation,True,NaN
4,1955319,Didn't have a chance to yesterday.,"We'd like to see what's going on, please conta...",follow_up_acknowledgment,True,NaN


In [ ]:
import time

predictions = []

for idx, row in df_golden.iterrows():
    message = row['clean_customer']
    try:
        predicted = classify_intent(message)
    except Exception as e:
        predicted = "ERROR"
        print(f"Error on row {idx}: {e}")
    predictions.append(predicted)
    time.sleep(1.5)
    if idx % 20 == 0:
        print(f"Processed {idx} of {len(df_golden)}")

df_golden['predicted_intent'] = predictions
print("Done!")

Processed 0 of 280
Processed 20 of 280
Processed 40 of 280
Processed 60 of 280
Processed 80 of 280
Processed 100 of 280
Processed 120 of 280
Processed 140 of 280
Processed 160 of 280
Processed 180 of 280
Processed 200 of 280
Processed 220 of 280
Processed 240 of 280
Processed 260 of 280
Done!


In [ ]:
import time

start = time.time()
result = classify_intent(df_golden['clean_customer'].iloc[0])
end = time.time()

print("Result:", result)
print("Time taken:", end - start, "seconds")

Result: delivery_delay
Time taken: 0.4821193218231201 seconds


In [ ]:
df_golden.to_csv('/content/drive/MyDrive/golden_set_with_predictions.csv', index=False)
print("Saved!")

Saved!


In [9]:
import pandas as pd

df_golden = pd.read_csv('/content/drive/MyDrive/golden_set_with_predictions.csv')

print(df_golden.shape)
df_golden.head()

(280, 7)


,tweet_id_customer,clean_customer,clean_amazon,intent,escalate,notes,predicted_intent
0,1366973,Two of the items don't have a delivery date ev...,I'm sorry for the wait! You'll receive a confi...,delivery_delay,True,NaN,delivery_delay
1,1254604,I haven’t been charged yet either,What delivery date were you given in your conf...,payment_issue,True,NaN,payment_issue
2,2697196,Merci de votre réponse aussi rapide. Je vais d...,"Je vous en prie, bonne soirée à vous aussi. ^BR",non_english,False,NaN,non_english
3,1456410,I have provided my details last night via chat...,We wouldn't be able to access your Amazon acco...,customer_service_escalation,True,NaN,follow_up_acknowledgment
4,1955319,Didn't have a chance to yesterday.,"We'd like to see what's going on, please conta...",follow_up_acknowledgment,True,NaN,non_english


In [10]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(df_golden['intent'], df_golden['predicted_intent'])
print(f"Overall accuracy: {accuracy:.2%}")

print(classification_report(df_golden['intent'], df_golden['predicted_intent']))

Overall accuracy: 51.43%
                                                                                                      precision    recall  f1-score   support

                                                                       account</think>account_access       0.00      0.00      0.00         0
                                                                                      account_access       0.30      0.25      0.27        12
                                                                                            customer       0.00      0.00      0.00         0
                                                                         customer_service_escalation       0.75      0.23      0.35        40
                                                                               damaged_or_wrong_item       0.83      0.77      0.80        13
                                                                                      delivery_delay       0.82      0.86 

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_

In [ ]:
print(df_golden['predicted_intent'].value_counts().to_string())

predicted_intent
delivery_delay                                                                                          83
general_other                                                                                           60
non_english                                                                                             23
payment_issue                                                                                           18
technical_issue                                                                                         16
customer_service_escalation                                                                             12
damaged_or_wrong_item                                                                                   12
follow_up_acknowledgment                                                                                11
account_access                                                                                          10
prime_subscription  

In [11]:
import re
from sklearn.metrics import accuracy_score, classification_report

INTENT_LABELS = [
    'delivery_delay', 'order_not_received', 'refund_return', 'damaged_or_wrong_item',
    'account_access', 'payment_issue', 'prime_subscription', 'product_content_request',
    'customer_service_escalation', 'technical_issue', 'follow_up_acknowledgment',
    'non_english', 'general_other'
]

def clean_prediction(raw):
    if pd.isna(raw):
        return 'unparseable'
    text = str(raw).strip()
    # If a </think> tag leaked through, keep only what comes after it
    if '</think>' in text:
        text = text.split('</think>')[-1].strip()
    # Exact match first
    if text in INTENT_LABELS:
        return text
    # Otherwise look for any known label appearing inside the text
    for label in INTENT_LABELS:
        if label in text:
            return label
    return 'unparseable'

df_golden['predicted_intent_clean'] = df_golden['predicted_intent'].apply(clean_prediction)

print('Unparseable rows:', (df_golden['predicted_intent_clean'] == 'unparseable').sum())

accuracy = accuracy_score(df_golden['intent'], df_golden['predicted_intent_clean'])
print(f"Overall accuracy (cleaned): {accuracy:.2%}")

print(classification_report(df_golden['intent'], df_golden['predicted_intent_clean'], zero_division=0))

Unparseable rows: 2
Overall accuracy (cleaned): 52.14%
                             precision    recall  f1-score   support

             account_access       0.36      0.33      0.35        12
customer_service_escalation       0.75      0.23      0.35        40
      damaged_or_wrong_item       0.83      0.77      0.80        13
             delivery_delay       0.82      0.87      0.85        79
   follow_up_acknowledgment       0.45      0.19      0.27        26
              general_other       0.15      0.53      0.23        17
                non_english       0.26      0.50      0.34        12
         order_not_received       0.44      0.31      0.36        13
              payment_issue       0.72      0.81      0.76        16
         prime_subscription       0.00      0.00      0.00         0
       prime_subscription         0.00      0.00      0.00         8
    product_content_request       0.00      0.00      0.00        22
              refund_return       0.80      0.8

In [12]:
# Strip whitespace from both the true labels and cleaned predictions
df_golden['intent'] = df_golden['intent'].astype(str).str.strip()
df_golden['predicted_intent_clean'] = df_golden['predicted_intent_clean'].astype(str).str.strip()

accuracy = accuracy_score(df_golden['intent'], df_golden['predicted_intent_clean'])
print(f"Overall accuracy (cleaned): {accuracy:.2%}")

print(classification_report(df_golden['intent'], df_golden['predicted_intent_clean'], zero_division=0))

Overall accuracy (cleaned): 53.93%
                             precision    recall  f1-score   support

             account_access       0.36      0.33      0.35        12
customer_service_escalation       0.75      0.23      0.35        40
      damaged_or_wrong_item       0.83      0.77      0.80        13
             delivery_delay       0.82      0.87      0.85        79
   follow_up_acknowledgment       0.45      0.19      0.27        26
              general_other       0.15      0.53      0.23        17
                non_english       0.26      0.50      0.34        12
         order_not_received       0.44      0.31      0.36        13
              payment_issue       0.72      0.81      0.76        16
         prime_subscription       0.50      0.62      0.56         8
    product_content_request       0.00      0.00      0.00        22
              refund_return       0.80      0.89      0.84         9
            technical_issue       0.56      0.69      0.62        1

In [13]:
mask = df_golden['intent'] == 'product_content_request'
subset = df_golden.loc[mask, ['clean_customer', 'intent', 'predicted_intent_clean']]

for i, row in subset.iterrows():
    print(f"MESSAGE: {row['clean_customer']}")
    print(f"TRUE LABEL: {row['intent']}   PREDICTED: {row['predicted_intent_clean']}")
    print("-" * 60)

MESSAGE: It’s a shame that doesn’t have a single movie of Chiranjeevi in its Telugu movies..pls acquire his all-time hits in your list
TRUE LABEL: product_content_request   PREDICTED: general_other
------------------------------------------------------------
MESSAGE: Hi , we were gifted this £50 voucher. We spent a while mulling over #BlackFridayWeekend deals that we might want..but seeing as it's so cold &amp; Xmas is near, we'd rather buy blankets for the homeless ☺️ Rather than us searching, what's the best deal you can make us?
TRUE LABEL: product_content_request   PREDICTED: general_other
------------------------------------------------------------
MESSAGE: bought echo in India, but echo assuming I am in US, how to install saavn &amp; Ola skills, not showing in skills search
TRUE LABEL: product_content_request   PREDICTED: technical_issue
------------------------------------------------------------
MESSAGE: grrrr trying to watch Vikings today on prime but it keeps saying band with

In [14]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    df_golden['intent'],
    df_golden['predicted_intent_clean'],
    labels=INTENT_LABELS + ['unparseable']
)

cm_df = pd.DataFrame(
    cm,
    index=INTENT_LABELS + ['unparseable'],
    columns=INTENT_LABELS + ['unparseable']
)

print(cm_df)

                             delivery_delay  order_not_received  \
delivery_delay                           69                   1   
order_not_received                        7                   4   
refund_return                             0                   0   
damaged_or_wrong_item                     0                   0   
account_access                            0                   0   
payment_issue                             0                   0   
prime_subscription                        0                   0   
product_content_request                   0                   0   
customer_service_escalation               2                   1   
technical_issue                           0                   0   
follow_up_acknowledgment                  2                   0   
non_english                               2                   1   
general_other                             2                   2   
unparseable                               0                   

In [16]:
non_english_texts = [
    "Des livres qui coûtent 100 euros, je comprends pas",
    "ei pessoal da Amazon, sabem dizer se a empresa vai expandir além de livros e eletronicos aqui no Brasil? Se sim, tem previsão?",
    "Si, pero no me han contestado",
    "La comodità dell'app di #Amazon che ti avvisa quando un oggetto della tua lista desideri è in offerta lampo 😍 #LeCoseFatteBene",
    "Io quando trovo offerte assurde su #asos o su #amazon"
]

mask = df_golden['clean_customer'].isin(non_english_texts)

# Handle the two tricky ones separately using a partial text match instead
mask = mask | df_golden['clean_customer'].str.contains('Büchern.*Wollte fragen', regex=True, na=False)
mask = mask | df_golden['clean_customer'].str.contains('Weihnachtskalender', na=False)

print("Rows matched:", mask.sum())

df_golden.loc[mask, 'intent'] = 'non_english'

accuracy = accuracy_score(df_golden['intent'], df_golden['predicted_intent_clean'])
print(f"Overall accuracy (final): {accuracy:.2%}")
print(classification_report(df_golden['intent'], df_golden['predicted_intent_clean'], zero_division=0))

df_golden.to_csv('/content/drive/MyDrive/golden_set_final.csv', index=False)

Rows matched: 7
Overall accuracy (final): 54.64%
                             precision    recall  f1-score   support

             account_access       0.36      0.33      0.35        12
customer_service_escalation       0.75      0.23      0.35        40
      damaged_or_wrong_item       0.83      0.77      0.80        13
             delivery_delay       0.82      0.87      0.85        79
   follow_up_acknowledgment       0.45      0.19      0.27        26
              general_other       0.15      0.53      0.23        17
                non_english       0.35      0.42      0.38        19
         order_not_received       0.44      0.31      0.36        13
              payment_issue       0.72      0.81      0.76        16
         prime_subscription       0.50      0.62      0.56         8
    product_content_request       0.00      0.00      0.00        15
              refund_return       0.80      0.89      0.84         9
            technical_issue       0.56      0.69     

In [18]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:

retrieval_corpus = df_english.dropna(subset=['clean_customer', 'clean_amazon']).reset_index(drop=True)
retrieval_corpus = retrieval_corpus.sample(n=15000, random_state=42).reset_index(drop=True)

print("Corpus size:", len(retrieval_corpus))



Corpus size: 15000


In [20]:
corpus_embeddings = embedder.encode(
    retrieval_corpus['clean_customer'].tolist(),
    show_progress_bar=True,
    batch_size=64
)

print("Embeddings shape:", corpus_embeddings.shape)

Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Embeddings shape: (15000, 384)


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_similar_replies(message, top_k=3):
    # Embed the new incoming message
    query_embedding = embedder.encode([message])

    # Compare against all corpus embeddings
    similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

    # Get indices of the top_k most similar past messages
    top_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'similar_customer_msg': retrieval_corpus.iloc[idx]['clean_customer'],
            'amazon_reply': retrieval_corpus.iloc[idx]['clean_amazon'],
            'similarity_score': similarities[idx]
        })
    return results

# Quick test
test_message = "my package still hasn't arrived and it's been a week"
results = retrieve_similar_replies(test_message)

for r in results:
    print(f"Similarity: {r['similarity_score']:.3f}")
    print(f"Similar past message: {r['similar_customer_msg']}")
    print(f"Amazon's reply: {r['amazon_reply']}")
    print("-" * 60)

Similarity: 0.794
Similar past message: Still haven't gotten my package
Amazon's reply: I'm sorry your package hasn't arrived! Can you tell us when it was expected and what the current tracking shows when you check here: ^EZ
------------------------------------------------------------
Similarity: 0.766
Similar past message: good morning. Apparently my package was delivered yesterday.. but I haven't received it. Can you help please?? Thanks
Amazon's reply: Sorry to hear that you haven't received it. This page will give you some handy tips in locating parcels scanned as delivered: ^MI
------------------------------------------------------------
Similarity: 0.762
Similar past message: I have not received my package which was to be delivered yesterday.
Amazon's reply: we'll be happy to help. 2/2 ^SH
------------------------------------------------------------


In [22]:
import numpy as np

np.save('/content/drive/MyDrive/corpus_embeddings.npy', corpus_embeddings)
retrieval_corpus.to_csv('/content/drive/MyDrive/retrieval_corpus.csv', index=False)

print("Saved!")

Saved!


In [23]:
def generate_reply(message, top_k=3):
    # Step 1: retrieve similar past resolutions
    similar_cases = retrieve_similar_replies(message, top_k=top_k)

    # Step 2: build a reference block from those real past replies
    examples_text = ""
    for i, case in enumerate(similar_cases, 1):
        examples_text += f"""Example {i}:
Customer: {case['similar_customer_msg']}
Amazon: {case['amazon_reply']}

"""

    prompt = f"""You are drafting a customer support reply for Amazon on Twitter, in Amazon's real tone and style.

Here are real past examples of similar customer messages and how Amazon actually replied:

{examples_text}
Now write a reply for this new customer message, in the same tone and style as the examples above. Keep it concise, empathetic, and helpful. Do not invent order numbers or details you don't have.

New customer message: "{message}"

Reply:"""

    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=150,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )
    return response.choices[0].message.content.strip()

# Quick test
test_message = "my package still hasn't arrived and it's been a week"
reply = generate_reply(test_message)
print(reply)

I'm sorry your package hasn't arrived! Can you tell us when it was expected and what the current tracking shows when you check here: ^EZ


In [25]:
test_messages = [
    "I want a refund for my order, it arrived completely broken",
    "I can't log into my account, it keeps saying my password is wrong even after I reset it",
    "Does the Kindle Paperwhite work outside the US or is it region locked?",
    "This is the third time I'm messaging about this and no one has helped me yet, I'm furious"
]

for msg in test_messages:
    print(f"CUSTOMER: {msg}")
    reply = generate_reply(msg)
    print(f"DRAFTED REPLY: {reply}")
    print("=" * 70)

CUSTOMER: I want a refund for my order, it arrived completely broken
DRAFTED REPLY: Oh no! I'm sorry to hear your order arrived broken. I can help with that. Please check your email for a return authorization, or let me know the order number and I'll get this started for you. ^BN
CUSTOMER: I can't log into my account, it keeps saying my password is wrong even after I reset it
DRAFTED REPLY: I'm sorry for the trouble you're having. You can contact us without having to log in here: ^EM
CUSTOMER: Does the Kindle Paperwhite work outside the US or is it region locked?
DRAFTED REPLY: Amazon: Yes, the Kindle Paperwhite works outside the US, but some features like the Kindle Store may be limited based on your location. For specific questions about regional restrictions, I recommend reaching out to our Kindle support team.
CUSTOMER: This is the third time I'm messaging about this and no one has helped me yet, I'm furious
DRAFTED REPLY: I understand that this experience has caused you disappoint

In [26]:
# Intents that should always escalate regardless of tone, human judgment needed
ALWAYS_ESCALATE_INTENTS = {"account_access", "refund_return", "customer_service_escalation"}

def decide_escalation(message, predicted_intent):
    # Rule 1: certain intents always need a human due to sensitivity or policy risk
    if predicted_intent in ALWAYS_ESCALATE_INTENTS:
        return {
            "escalate": True,
            "reason": f"Intent '{predicted_intent}' involves account security or money, requires human review by policy."
        }

    # Rule 2: obvious frustration signals, catch repeated contact / anger cheaply without an LLM call
    frustration_keywords = ["third time", "again", "still no", "no one has helped", "furious", "ridiculous", "unacceptable"]
    if any(kw in message.lower() for kw in frustration_keywords):
        return {
            "escalate": True,
            "reason": "Message contains strong frustration or repeated-contact language, flagged for human review."
        }

    # Otherwise, ask the LLM to judge based on content and tone
    prompt = f"""You are deciding whether a customer support message should be auto-handled by an AI agent or escalated to a human agent.

Message: "{message}"
Predicted intent: {predicted_intent}

Escalate to a human if the message involves anger, confusion, high financial stakes, safety issues, or anything an AI shouldn't decide alone.
Otherwise, mark it as safe to auto-handle.

Reply in exactly this format:
DECISION: [escalate or auto-handle]
REASON: [one short sentence]"""

    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=60,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )
    output = response.choices[0].message.content.strip()

    escalate = "escalate" in output.lower().split("reason")[0].lower()
    reason = output.split("REASON:")[-1].strip() if "REASON:" in output else output

    return {"escalate": escalate, "reason": reason}

# Quick test
print(decide_escalation("my package still hasn't arrived and it's been a week", "delivery_delay"))
print(decide_escalation("I can't log into my account, it keeps saying my password is wrong", "account_access"))

{'escalate': False, 'reason': 'The message expresses a common delivery issue without signs of anger, confusion, or high financial stakes.'}
{'escalate': True, 'reason': "Intent 'account_access' involves account security or money, requires human review by policy."}


In [27]:
import time

escalate_predictions = []
reason_predictions = []

for i, row in df_golden.iterrows():

    try:
        result = decide_escalation(
            row['clean_customer'],
            row['predicted_intent_clean']
        )

        escalate_predictions.append(result['escalate'])
        reason_predictions.append(result['reason'])

    except Exception as e:
        escalate_predictions.append("ERROR")
        reason_predictions.append(str(e))

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1} / {len(df_golden)}")

    time.sleep(1.5)

df_golden['predicted_escalate'] = escalate_predictions
df_golden['escalate_reason'] = reason_predictions

print("Done!")

Processed 20 / 280
Processed 40 / 280
Processed 60 / 280
Processed 80 / 280
Processed 100 / 280
Processed 120 / 280
Processed 140 / 280
Processed 160 / 280
Processed 180 / 280
Processed 200 / 280
Processed 220 / 280
Processed 240 / 280
Processed 260 / 280
Processed 280 / 280
Done!


In [30]:
df_golden.to_csv('/content/drive/MyDrive/golden_set_with_escalation.csv', index=False)

from sklearn.metrics import accuracy_score, classification_report

escalation_accuracy = accuracy_score(df_golden['escalate'], df_golden['predicted_escalate'])
print(f"Escalation decision accuracy: {escalation_accuracy:.2%}")

print(classification_report(df_golden['escalate'], df_golden['predicted_escalate']))

ValueError: Classification metrics can't handle a mix of binary and unknown targets

In [31]:
print(df_golden['escalate'].unique())
print(df_golden['escalate'].dtype)
print()
print(df_golden['predicted_escalate'].unique())
print(df_golden['predicted_escalate'].dtype)

[ True False]
bool

[False True 'ERROR']
object


In [32]:
# Keep only rows where the escalation prediction succeeded
valid_mask = df_golden['predicted_escalate'] != 'ERROR'
print("Rows with errors excluded:", (~valid_mask).sum())

df_valid = df_golden[valid_mask].copy()

# Make sure both columns are proper booleans for comparison
df_valid['predicted_escalate'] = df_valid['predicted_escalate'].astype(bool)

escalation_accuracy = accuracy_score(df_valid['escalate'], df_valid['predicted_escalate'])
print(f"Escalation decision accuracy: {escalation_accuracy:.2%}")

print(classification_report(df_valid['escalate'], df_valid['predicted_escalate']))

Rows with errors excluded: 2
Escalation decision accuracy: 69.42%
              precision    recall  f1-score   support

       False       0.64      0.85      0.73       136
        True       0.79      0.54      0.64       142

    accuracy                           0.69       278
   macro avg       0.72      0.70      0.69       278
weighted avg       0.72      0.69      0.69       278



In [33]:
from sklearn.metrics import accuracy_score

# Baseline 1: trivial baseline, always predict the single most common intent
most_common_intent = df_golden['intent'].mode()[0]
print("Most common intent:", most_common_intent)

trivial_predictions = [most_common_intent] * len(df_golden)
trivial_accuracy = accuracy_score(df_golden['intent'], trivial_predictions)
print(f"Trivial baseline accuracy: {trivial_accuracy:.2%}")

print()

# Baseline 2: simple keyword-matching baseline
KEYWORD_RULES = {
    "delivery_delay": ["hasn't arrived", "still waiting", "late", "delayed", "where is my", "not received yet"],
    "order_not_received": ["never received", "did not receive", "haven't gotten"],
    "refund_return": ["refund", "return", "money back"],
    "damaged_or_wrong_item": ["broken", "damaged", "wrong item", "defective"],
    "account_access": ["can't log in", "login", "password", "locked out"],
    "payment_issue": ["charged", "payment", "billing", "double charge"],
    "prime_subscription": ["prime", "subscription", "membership"],
    "product_content_request": ["available", "release date", "season", "when will"],
    "technical_issue": ["app", "error", "not working", "bug", "crash"],
}

def keyword_baseline(message):
    message_lower = str(message).lower()
    for intent, keywords in KEYWORD_RULES.items():
        if any(kw in message_lower for kw in keywords):
            return intent
    return "general_other"

keyword_predictions = df_golden['clean_customer'].apply(keyword_baseline)
keyword_accuracy = accuracy_score(df_golden['intent'], keyword_predictions)
print(f"Keyword baseline accuracy: {keyword_accuracy:.2%}")

Most common intent: delivery_delay
Trivial baseline accuracy: 28.21%

Keyword baseline accuracy: 19.29%
